In [3]:
# ================================
# 1. Import thư viện
# ================================
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ================================
# 2. Dataset
# ================================
COLOR_MAP = {
    (255, 255, 255): 0,  # Background
    (0, 0, 255): 1,      # Non-tumor Tissue
    (255, 0, 0): 2,      # Fibrosis/Hyalinization
    (0, 255, 0): 3       # Viable Tumor
}

def rgb_to_mask(rgb_img):
    rgb = np.array(rgb_img)
    mask = np.zeros((rgb.shape[0], rgb.shape[1]), dtype=np.uint8)
    for color, idx in COLOR_MAP.items():
        matches = np.all(rgb == np.array(color), axis=-1)
        mask[matches] = idx
    return mask

class HistologyDataset(Dataset):
    def __init__(self, img_dir, mask_dir, patch_size=256, stride=256, transform=None):
        self.img_paths = sorted(glob(os.path.join(img_dir, "*.png")))
        self.mask_paths = sorted(glob(os.path.join(mask_dir, "*.png")))
        self.patch_size = patch_size
        self.stride = stride
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("RGB")

        img = img.resize((self.patch_size, self.patch_size), Image.BILINEAR)
        mask = mask.resize((self.patch_size, self.patch_size), Image.NEAREST)

        # Convert ảnh sang tensor
        img = transforms.ToTensor()(img)

        # Convert mask RGB → index mask
        mask = rgb_to_mask(mask)
        mask = torch.tensor(mask, dtype=torch.long)  # [H,W]

        return img, mask


# ================================
# 3. ResUNet model
# ================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ResUNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.enc1 = ConvBlock(3, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.center = ConvBlock(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = ConvBlock(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(128, 64)

        self.final = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        c = self.center(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(c), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.final(d1)

# ================================
# 4. Metrics
# ================================
def dice_coef(pred, target, eps=1e-6):
    pred = torch.argmax(pred, dim=1)
    inter = (pred & target).float().sum((1,2))
    union = pred.float().sum((1,2)) + target.float().sum((1,2))
    dice = (2 * inter + eps) / (union + eps)
    return dice.mean().item()

def iou_score(pred, target, eps=1e-6):
    pred = torch.argmax(pred, dim=1)
    inter = (pred & target).float().sum((1,2))
    union = pred.float().sum((1,2)) + target.float().sum((1,2)) - inter
    iou = (inter + eps) / (union + eps)
    return iou.mean().item()

# ================================
# 5. Training function
# ================================
def train_model(img_dir, mask_dir, num_classes=4, epochs=50, batch_size=4, lr=1e-3, patience=5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    dataset = HistologyDataset(img_dir, mask_dir, patch_size=256)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = ResUNet(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    best_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_dice, epoch_iou = 0, 0, 0

        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_dice += dice_coef(outputs.detach(), masks)
            epoch_iou += iou_score(outputs.detach(), masks)

        epoch_loss /= len(loader)
        epoch_dice /= len(loader)
        epoch_iou /= len(loader)

        scheduler.step(epoch_loss)

        print(f"[{epoch+1}/{epochs}] Loss={epoch_loss:.4f} Dice={epoch_dice:.4f} IoU={epoch_iou:.4f} LR={optimizer.param_groups[0]['lr']:.6f}")

        # Early stopping + checkpoint
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("  🔥 Saved best model")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("⏹️ Early stopping triggered")
                break

    print("Training complete. Best loss:", best_loss)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
import os
import random
import numpy as np

# 1. Python built-in random
random.seed(42)

# 2. NumPy
np.random.seed(42)

# 3. PyTorch
import torch
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# 4. TensorFlow
import tensorflow as tf
tf.random.set_seed(42)

# 5. Make hash-based operations deterministic
os.environ["PYTHONHASHSEED"] = "42"

In [8]:
# ================================
# 6. Run training
# ================================
import os
import random
import numpy as np

# 1. Python built-in random
random.seed(42)

# 2. NumPy
np.random.seed(42)

# 3. PyTorch
import torch
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# 4. TensorFlow
import tensorflow as tf
tf.random.set_seed(42)

# 5. Make hash-based operations deterministic
os.environ["PYTHONHASHSEED"] = "42"

if __name__ == "__main__":
    img_dir = "/kaggle/input/iiioppoopop/train/images"
    mask_dir = "/kaggle/input/iiioppoopop/train/labels"
    train_model(img_dir, mask_dir, num_classes=4, epochs=30, batch_size=4, lr=1e-3, patience=5)
# ================================
# Load trained model for inference
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate the model with correct num_classes
model = ResUNet(num_classes=4).to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

Using device: cuda
[1/30] Loss=0.9914 Dice=0.4820 IoU=0.3461 LR=0.001000
  🔥 Saved best model
[2/30] Loss=0.7843 Dice=0.5481 IoU=0.4091 LR=0.001000
  🔥 Saved best model
[3/30] Loss=0.7720 Dice=0.5229 IoU=0.3817 LR=0.001000
  🔥 Saved best model
[4/30] Loss=0.7071 Dice=0.5667 IoU=0.4318 LR=0.001000
  🔥 Saved best model
[5/30] Loss=0.7154 Dice=0.5546 IoU=0.4157 LR=0.001000
[6/30] Loss=0.6909 Dice=0.5454 IoU=0.4070 LR=0.001000
  🔥 Saved best model
[7/30] Loss=0.6736 Dice=0.5692 IoU=0.4343 LR=0.001000
  🔥 Saved best model
[8/30] Loss=0.7070 Dice=0.5537 IoU=0.4151 LR=0.001000
[9/30] Loss=0.6746 Dice=0.5624 IoU=0.4266 LR=0.001000
[10/30] Loss=0.6512 Dice=0.5657 IoU=0.4345 LR=0.001000
  🔥 Saved best model
[11/30] Loss=0.6429 Dice=0.5747 IoU=0.4387 LR=0.001000
  🔥 Saved best model
[12/30] Loss=0.6581 Dice=0.5633 IoU=0.4276 LR=0.001000
[13/30] Loss=0.6555 Dice=0.5612 IoU=0.4250 LR=0.001000
[14/30] Loss=0.6710 Dice=0.5547 IoU=0.4186 LR=0.001000
[15/30] Loss=0.6828 Dice=0.5645 IoU=0.4231 LR=0.0005

ResUNet(
  (enc1): ConvBlock(
    (conv): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): ConvBlock(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): ConvBlock(
    (conv): Sequential(
      (0): Conv2d(128, 256, kernel_size=

**#Load best_model.pth# **# 

In [ ]:
# Recreate the model with correct num_classes
model = ResUNet(num_classes=4).to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

In [9]:
class InferenceDataset(Dataset):
    def __init__(self, img_dir, patch_size=256, transform=None):
        self.img_paths = sorted(glob(os.path.join(img_dir, "*.*")))
        self.patch_size = patch_size
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img = Image.open(self.img_paths[idx]).convert("RGB")
        img_resized = img.resize((self.patch_size, self.patch_size), Image.BILINEAR)
        img_tensor = transforms.ToTensor()(img_resized)
        return img_tensor, os.path.basename(self.img_paths[idx]), img.size  # (W,H)


In [12]:
val_img_dir = "/kaggle/input/anonymous-2"
val_dataset = InferenceDataset(val_img_dir, patch_size=256)

out_dir = "predictions"
os.makedirs(out_dir, exist_ok=True)
INDEX_TO_COLOR = {
    0: (255,255,255),
    1: (0,0,255),
    2: (255,0,0),
    3: (0,255,0),
}

def decode_mask(mask):
    if torch.is_tensor(mask):
        mask = mask.cpu().numpy()
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for idx, color in INDEX_TO_COLOR.items():
        color_mask[mask == idx] = color
    return color_mask
with torch.no_grad():
    for idx in range(len(val_dataset)):
        img_tensor, fname, orig_size = val_dataset[idx]
        orig_w, orig_h = orig_size

        img = img_tensor.unsqueeze(0).to(device)

        # forward pass
        output = model(img)
        pred = torch.argmax(output, dim=1).squeeze(0)

        # resize prediction back to original size
        pred_resized = F.interpolate(
            pred.unsqueeze(0).unsqueeze(0).float(),
            size=(orig_h, orig_w),
            mode="nearest"
        ).squeeze().long().cpu().numpy()

        # decode to color
        pred_color = decode_mask(pred_resized)

        # save
        out_path = os.path.join(out_dir, fname)
        Image.fromarray(pred_color).save(out_path)
        print("Saved:", out_path)

Saved: predictions/Case_0.png
Saved: predictions/Case_1.png
Saved: predictions/Case_10.png
Saved: predictions/Case_11.png
Saved: predictions/Case_12.png
Saved: predictions/Case_13.png
Saved: predictions/Case_14.png
Saved: predictions/Case_15.png
Saved: predictions/Case_16.png
Saved: predictions/Case_17.png
Saved: predictions/Case_18.png
Saved: predictions/Case_19.png
Saved: predictions/Case_2.png
Saved: predictions/Case_3.png
Saved: predictions/Case_4.png
Saved: predictions/Case_5.png
Saved: predictions/Case_6.png
Saved: predictions/Case_7.png
Saved: predictions/Case_8.png
Saved: predictions/Case_9.png


In [13]:
import zipfile
# zip everything
zip_path = "predictions.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in os.listdir(out_dir):
        zf.write(os.path.join(out_dir, file), arcname=file)

print("✅ Zipped predictions:", zip_path)

✅ Zipped predictions: predictions.zip
